# Bundle Authoring Guide Notebook

Mirrors the [Bundle Authoring Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_authoring_guide/). Each section follows the same headings as the guide.

Run from `docs/assets/` so the relative path to `MODELS/POST82.yaml` resolves.

In [21]:
import io

import numpy as np
import pandas as pd
from numpy import array, float64

from SymbolicDSGE import BundleBuilder, DSGESolver, Estimator, ModelParser, Shock
from SymbolicDSGE.bayesian import make_prior
from SymbolicDSGE.bundle import ShockGeneration, SimSpec

from SymbolicDSGE.monte_carlo import MCPipeline
from SymbolicDSGE.monte_carlo.operations import core as c, tests as t

## Solve a reference model

In [22]:
# Create the Bundle
bundle = BundleBuilder(created_by="experiment-1")

parser = ModelParser("../../MODELS/POST82.yaml")
model, kalman = parser.get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(linearize=False)
sol = solver.solve(
    compiled,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

# Add the model to the bundle
bundle.add_model(
    "reference",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"steady_state": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

# The `dgp` passed to `mc_pipeline.run(...)` is a runtime-only argument; it is
# NOT persisted by `add_mc`. To make `loaded.dgp` resolve, the DGP model must be
# bundled explicitly under role "dgp". Here the experiment uses the same model as
# its DGP, so we ship the same YAML (in a misspecification study this would be a
# different model's source).
bundle.add_model(
    "dgp",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"steady_state": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

## Specify the estimation tab

`EstimationSpec.from_targets` builds the spec from just the parameter names — `estimate=True` is flagged for each automatically. The synthetic `OptimizationResultMeta` below stands in for a real result so the demo bundle still ships a populated estimation tab; the fast-path cells later replace this manual construction with a live `Estimator` run.

In [23]:
# estimation_spec = EstimationSpec.from_targets(
#     ["psi_pi", "psi_x"],
#     method="map",
#     initial={"psi_pi": 1.5, "psi_x": 0.5},
#     bounds={"psi_pi": (1.0, 3.0), "psi_x": (0.0, 1.0)},
#     priors={
#         "psi_pi": PriorSpec(
#             distribution="normal", parameters={"loc": 1.5, "scale": 0.25}
#         ),
#         "psi_x": PriorSpec(
#             distribution="normal", parameters={"loc": 0.5, "scale": 0.2}
#         ),
#     },
#     observables=["Infl", "Rate"],
#     method_kwargs={"options": {"maxiter": 50}},
# )

# rng = np.random.default_rng(0)
# observed = rng.standard_normal((40, 2))
# estimation_result_meta = OptimizationResultMeta(
#     kind="map",
#     theta={"psi_pi": 1.48, "psi_x": 0.55},
#     success=True,
#     message="Optimization converged.",
#     fun=-87.3,
#     loglik=-85.1,
#     logprior=-2.2,
#     logpost=-87.3,
#     nfev=124,
#     nit=14,
# )

priors = {
    "psi_pi": make_prior(
        distribution="normal",
        parameters={"mean": 1.5, "std": 0.25},
        transform="identity",
    ),
    "psi_x": make_prior(
        distribution="normal",
        parameters={"mean": 0.5, "std": 0.2},
        transform="identity",
    ),
}

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 3))
estim = Estimator(
    solver=solver,
    compiled=compiled,
    observables=["OutGap", "Infl", "Rate"],
    y=observed,
    priors=priors,
)
res = estim.mcmc(
    n_draws=1000,
    burn_in=200,
    thin=2,
)

MCMC sampling concluded in 1.38 seconds with 1594.03 iterations per second.
[Estimator:mcmc] BK stability warnings encountered during search: 0


In [24]:
# Add the estimation to the bundle with results
bundle.add_estimation(
    source=estim,
    result=res,
)

## Build a Monte Carlo pipeline

In [25]:
# mc_pipeline = PipelineSpec(
#     nodes=[
#         NodeSpec(
#             id="sim",
#             step_type="simulation",
#             name="DGP Simulation",
#             params={"T": 100},
#         ),
#         NodeSpec(
#             id="jb",
#             step_type="jarque_bera",
#             name="Normality Check",
#             params={"source": "observables", "column": 0},
#         ),
#     ],
#     edges=[EdgeSpec(source="sim", target="jb")],
# )

# Pass the `Shock` *instances* (not `.shock_generator()`): the pipeline clones
# them per replication with seeded offsets, and they serialize into the bundle.
gz_shock = Shock(T=200, seed=42, multivar=True, dist="norm")
r_shock = Shock(T=200, seed=42, multivar=False, dist="norm")

mc_pipeline = MCPipeline(
    [
        c.simulation_step(T=200, shocks={"g,z": gz_shock, "r": r_shock}),
        t.jarque_bera_test_step("jb_test", source="observables", column=0),
    ]
)
mc_res = mc_pipeline.run(
    reference=sol,
    dgp=sol,
    n_rep=1000,
    retain_payloads=False,
    retain_contexts=True,
    verbosity=2,
)

datagen concluded successfully with 1333.81 it/s.
jb_test concluded successfully with 37106.58 it/s.


In [26]:
# Add the Monte Carlo pipeline to the bundle
bundle.add_mc(
    pipeline=mc_pipeline,
    result=mc_res,
)

## Specify a simulation prefill

In [27]:
simulation = SimSpec(
    role="reference",
    T=200,
    observables=True,
    shock_scale=1.0,
    shock_generation=ShockGeneration(
        dist="norm",
        seed=42,
        loc=0.0,
    ),
)

# Add the simulation spec to the bundle
bundle.set_simulation(simulation)

## Add raw data

In [28]:
# Make text data
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buf = io.StringIO()
aux.to_csv(csv_buf, index=False)

bundle.add_raw_data(
    name="auxiliary_series",
    data=csv_buf.getvalue(),
)

## Write the bundle to disk

In [29]:
bundle.write("experiment-1.sdsge")

WindowsPath('experiment-1.sdsge')

## Inspect the result

In [30]:
!unzip -l experiment-1.sdsge

Archive:  experiment-1.sdsge
  Length      Date    Time    Name
---------  ---------- -----   ----
     3122  16-06-2026 14:26   manifest.json
     2599  16-06-2026 14:26   model/reference.yaml
     2599  16-06-2026 14:26   model/dgp.yaml
      973  16-06-2026 14:26   estimation/spec.json
      420  16-06-2026 14:26   estimation/result.json
     1939  16-06-2026 14:26   estimation/observed.parquet
    19943  16-06-2026 14:26   estimation/posterior.parquet
      832  16-06-2026 14:26   montecarlo/pipeline.json
     4475  16-06-2026 14:26   montecarlo/result.json
    15645  16-06-2026 14:26   montecarlo/traces.parquet
     1282  16-06-2026 14:26   data/auxiliary_series.parquet
---------                     -------
    53829                     11 files


In [31]:
# File size of the bundle in KB
import os

sizeof = round(os.path.getsize("experiment-1.sdsge") / 1024)
print(f"Bundle size: {sizeof} KB")

Bundle size: 44 KB
